# Real vLLM inference with TRITON_ATTN

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Loads `Qwen/Qwen2.5-0.5B-Instruct` (public, ~1 GB fp16) from Hugging Face, runs short-decode and ~1.2k-token prefill generations at TP=1 and TP=2, checks determinism, and **greps vLLM's own log** for the selected backend. Set `HF_TOKEN` as a Kaggle secret only for gated models.

## 1. Setup — locate the release bundle

In [ ]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Explicit release identity: only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted
# (an older rc is never silently selected). Source 1: the attached Kaggle dataset. Source 2 (nothing attached): the Hugging Face CDN.
RELEASE = "stokastisk-0.1.0-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading it from Hugging Face (needs Internet ON) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE)

## 2. Install into this fresh Kaggle session (idempotent, `--no-deps`, SHA256-verified, protected packages untouched)

Kaggle gives every notebook a new session: what notebook 02 installed does not exist here. Internet must be ON.

In [ ]:
import subprocess, sys, os
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)],
                   env=dict(os.environ, STOKASTISK_OUT="/kaggle/working/stokastisk-acceptance/prereq-install"))
assert r.returncode == 0, "prerequisite install failed (see output above); do not continue"

## 3. Provenance — every executed script and wheel must match `manifests/SHA256SUMS`

In [ ]:
import hashlib, glob
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
print("bundle root     :", BUNDLE)
print("scripts/kaggle  :", SCRIPTS)
for s in sorted(SCRIPTS.glob("*.py")): print(f"{sha(s)[:16]}  {s.name}")
for w in sorted((BUNDLE / "wheels").glob("*.whl")): print(f"{sha(w)}  {w.name}")
expected = {l.split(None, 1)[1].strip(): l.split()[0] for l in (BUNDLE / "manifests" / "SHA256SUMS").read_text().splitlines()}
for rel in [f"scripts/kaggle/{s.name}" for s in SCRIPTS.glob("*.py")] + [f"wheels/{w.name}" for w in (BUNDLE / "wheels").glob("*.whl")]:
    assert expected.get(rel) == sha(BUNDLE / rel), f"{rel} does not match manifests/SHA256SUMS (stale or modified file)"
print("all executed scripts and wheels match manifests/SHA256SUMS")

## 4. The installed packages must be byte-identical to the bundle's wheels

In [ ]:
import importlib, importlib.metadata as md, zipfile, hashlib
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]
    mod = importlib.import_module(pkg)
    root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)} imported from {root}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}; uninstall it and re-run the install cell"

## 5. Run Qwen2.5-0.5B-Instruct at TP=1 and TP=2

In [ ]:
RC = {}
env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""), VLLM_LOGGING_LEVEL="INFO")
os.makedirs("/kaggle/working/stokastisk-acceptance", exist_ok=True)
for tp in (1, 2):
    with open(f"/kaggle/working/stokastisk-acceptance/vllm_tp{tp}.log", "w") as f:
        RC[f"05_inference_tp{tp}"] = subprocess.run([sys.executable, str(SCRIPTS / "05_inference.py"), str(tp), "Qwen/Qwen2.5-0.5B-Instruct"],
                                                   cwd=SCRIPTS, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    print("TP", tp, "rc", RC[f"05_inference_tp{tp}"], "-- log: vllm_tp%d.log (a nonzero rc is judged in the verdict cell, after the evidence is packaged)" % tp)

## 6. Prove from vLLM's own logs that `TRITON_ATTN` was selected

In [ ]:
import subprocess, sys, os
def run_stage(script, *args):
    """Run a bundled stage script; returns its exit code (a failure is asserted in the final verdict cell, after evidence is packaged)."""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS,
                       env=dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", "")))
    print(f"{script} exit code: {r.returncode}")
    return r.returncode

In [ ]:
RC["05_backend_evidence"] = run_stage("05_backend_evidence.py", "1", "2")
for tp in (1, 2):
    f = pathlib.Path(f"/kaggle/working/stokastisk-acceptance/05_inference_tp{tp}.json")
    print(f"--- {f.name}:", (f.read_text()[:1500] if f.is_file() else "NOT WRITTEN"))

## Package the evidence

Creates `/kaggle/working/05-stokastisk-acceptance.zip` (download it from the Output tab).

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/05-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

## Verdict — this cell fails the notebook if any stage failed

In [ ]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)